# Fraud Detection — Exploratory Data Analysis
**Analyst:** Sagar Kandelkar | **Date:** September 2026
**Data:** Synthetic transaction dataset for portfolio case study

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/transactions.csv')
alerts = pd.read_csv('../data/alerts.csv')
print('Transactions shape:', df.shape)
print('Alerts shape:', alerts.shape)
df.head()

## 2. Data Profile

In [ ]:
print(df.info())
print('\n--- Numeric Summary ---')
print(df.describe())

## 3. Fraud Distribution

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_rate = df['is_fraud'].mean() * 100
print(f'Fraud Rate: {fraud_rate:.1f}%')

fig, ax = plt.subplots()
colors = ['#2563eb', '#dc2626']
fraud_counts.plot(kind='bar', color=colors, ax=ax)
ax.set_title('Fraud vs Genuine Transactions')
ax.set_xlabel('Is Fraud')
ax.set_ylabel('Count')
ax.set_xticklabels(['Genuine', 'Fraud'], rotation=0)
plt.tight_layout()
plt.show()

## 4. Risk Score Distribution

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(x='is_fraud', y='risk_score', data=df, palette=['#2563eb', '#dc2626'], ax=ax)
ax.set_title('Risk Score by Fraud Status')
ax.set_xticklabels(['Genuine', 'Fraud'])
plt.tight_layout()
plt.show()

print(df.groupby('is_fraud')['risk_score'].mean())

## 5. Transaction Amount Analysis

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(x='is_fraud', y='amount', data=df, palette=['#2563eb', '#dc2626'], ax=ax)
ax.set_title('Transaction Amount by Fraud Status')
ax.set_xticklabels(['Genuine', 'Fraud'])
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print('Average amount:')
print(df.groupby('is_fraud')['amount'].mean())

## 6. Hourly Fraud Pattern

In [ ]:
df['datetime'] = pd.to_datetime(df['transaction_date'] + ' ' + df['transaction_time'])
df['hour'] = df['datetime'].dt.hour

hourly = df.groupby(['hour', 'is_fraud']).size().unstack(fill_value=0)
hourly.plot(kind='bar', stacked=True, color=['#2563eb', '#dc2626'])
plt.title('Transactions by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.legend(['Genuine', 'Fraud'])
plt.tight_layout()
plt.show()

off_hours = df[df['hour'].isin([0,1,2,3,4,5,22,23])]
print(f'Off-hours fraud rate: {off_hours["is_fraud"].mean()*100:.1f}%')

## 7. Channel Risk Analysis

In [ ]:
channel_fraud = df.groupby('channel')['is_fraud'].agg(['count', 'sum', 'mean']).reset_index()
channel_fraud.columns = ['channel', 'total', 'fraud_count', 'fraud_rate']
channel_fraud['fraud_rate'] = channel_fraud['fraud_rate'] * 100

fig, ax1 = plt.subplots()
x = np.arange(len(channel_fraud))
bars = ax1.bar(x, channel_fraud['total'], color='#2563eb', alpha=0.7, label='Total')
bars2 = ax1.bar(x, channel_fraud['fraud_count'], color='#dc2626', label='Fraud')
ax1.set_xticks(x)
ax1.set_xticklabels(channel_fraud['channel'], rotation=45)
ax1.set_title('Transactions by Channel')
ax1.legend()
plt.tight_layout()
plt.show()

print(channel_fraud[['channel', 'fraud_rate']])

## 8. Alert Triggers Breakdown

In [ ]:
# Expand alert triggers
triggers = df['alert_trigger'].dropna().str.split(',').explode().str.strip()
trigger_counts = triggers.value_counts()

trigger_counts.plot(kind='barh', color='#dc2626')
plt.title('Most Common Alert Triggers')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 9. Correlation Heatmap

In [ ]:
numeric_cols = ['amount', 'risk_score', 'is_fraud', 'velocity_count']
corr = df[numeric_cols].corr()

sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 10. Key Insights & Recommendations

1. **Fraud Rate:** ~28% fraud rate in this sample indicates targeted monitoring is critical.
2. **Risk Score:** Fraudulent transactions have significantly higher risk scores (avg ~85 vs ~35 genuine).
3. **Amount Pattern:** Fraud transactions show higher amounts; implement dynamic thresholds.
4. **Off-Hours:** Night-time transactions (10 PM – 5 AM) show elevated fraud rates.
5. **Channel Risk:** Online channels (Internet Banking, UPI) carry higher fraud risk than branch.
6. **Top Triggers:** Velocity rules and off-hours activity are the most frequent alert triggers.

### Recommendations
- Increase scrutiny for transactions > ₹50,000 in online channels
- Apply stricter velocity limits during off-hours
- Implement step-up authentication for high-risk score transactions
- Focus rule tuning on velocity and off-hours patterns